# Fine-tune ConvNeXt note classifier

In [1]:
# Colab setup. Run from the repo root after uploading/cloning this project.
# !pip install -r requirements.txt

# ==== Colab project setup ====

from pathlib import Path
import sys
import os

REPO_URL = "https://github.com/vadatta/spectogram-audio.git"
REPO_NAME = "spectogram-audio"

# Go to Colab workspace
%cd /content

# Clone repo if it does not exist yet
if not Path(REPO_NAME).exists():
    !git clone {REPO_URL}

# Enter project folder
%cd /content/{REPO_NAME}

# Pull latest changes from GitHub
!git pull

# Add project root to Python import path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Files:", os.listdir(PROJECT_ROOT))
!ls

/content
Cloning into 'spectogram-audio'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 54 (delta 21), reused 36 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 38.13 KiB | 9.53 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/spectogram-audio
Already up to date.
Project root: /content/spectogram-audio
Files: ['nsynth_pipeline', 'notebooks', '.git', '.DS_Store', '.gitignore', '.vscode', 'requirements.txt']
notebooks  nsynth_pipeline  requirements.txt


In [2]:
!python -m pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 14.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 20.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [3]:
import sys
print(sys.executable)

/usr/bin/python3


In [4]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from nsynth_pipeline.convexnet import freeze_module, load_convnext_backbone
from nsynth_pipeline.data import MelConfig, NOTE_NAMES, make_nsynth_loaders
from nsynth_pipeline.models import ConvNeXtNoteClassifier
from nsynth_pipeline.train_utils import move_batch, save_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [5]:
DATASET_NAME = "jg583/NSynth"
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 4
# Raw pitch is used for reconstruction conditioning, so weight it more heavily.
PITCH_LOSS_WEIGHT = 2.0

# Set these to small integers for smoke tests; keep None for full train/test splits.
MAX_TRAIN_ITEMS = 2048
MAX_TEST_ITEMS = 512

CHECKPOINT_PATH = "checkpoints/convnext_pitch_classifier.pt"
mel_config = MelConfig()

In [6]:
# In Colab, this stores the Hugging Face dataset cache on Google Drive so it persists across runtimes.
HF_CACHE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    HF_CACHE_DIR = "/content/drive/MyDrive/hf_datasets_cache"
except ModuleNotFoundError:
    print("Not running in Colab; using the local Hugging Face cache.")

print("HF cache dir:", HF_CACHE_DIR)

Mounted at /content/drive
HF cache dir: /content/drive/MyDrive/hf_datasets_cache


In [8]:
!rm -rf "/content/drive/MyDrive/hf_datasets_cache"

^C


In [7]:
loaders = make_nsynth_loaders(
    dataset_name=DATASET_NAME,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_train_items=MAX_TRAIN_ITEMS,
    max_test_items=MAX_TEST_ITEMS,
    mel_config=mel_config,
    cache_dir=HF_CACHE_DIR,
)
batch = next(iter(loaders["train"]))
batch["spectrogram"].shape, batch["note_class"][:8], [NOTE_NAMES[i] for i in batch["note_class"][:8].tolist()]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/289205 [00:00<?, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

In [7]:
backbone = load_convnext_backbone()
freeze_module(backbone)

model = ConvNeXtNoteClassifier(backbone).to(device)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:05<00:00, 61.3MB/s]


## Training code

In [8]:
def train_classifier_one_epoch(model, loader, optimizer, device, pitch_loss_weight=2.0):
    model.train()
    if not any(parameter.requires_grad for parameter in model.backbone.parameters()):
        model.backbone.eval()

    running_loss = 0.0
    running_note_loss = 0.0
    running_pitch_loss = 0.0
    note_correct = 0
    pitch_correct = 0
    total = 0

    for batch in tqdm(loader, desc="classifier train"):
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch["spectrogram"])
        note_loss = F.cross_entropy(outputs["note_logits"], batch["note_class"])
        pitch_loss = F.cross_entropy(outputs["pitch_logits"], batch["pitch"])
        loss = note_loss + pitch_loss_weight * pitch_loss
        loss.backward()
        optimizer.step()

        batch_size = batch["note_class"].shape[0]
        running_loss += loss.item() * batch_size
        running_note_loss += note_loss.item() * batch_size
        running_pitch_loss += pitch_loss.item() * batch_size
        note_correct += (outputs["note_logits"].argmax(dim=-1) == batch["note_class"]).sum().item()
        pitch_correct += (outputs["pitch_logits"].argmax(dim=-1) == batch["pitch"]).sum().item()
        total += batch_size

    return {
        "loss": running_loss / total,
        "note_loss": running_note_loss / total,
        "pitch_loss": running_pitch_loss / total,
        "note_accuracy": note_correct / total,
        "pitch_accuracy": pitch_correct / total,
    }


@torch.no_grad()
def evaluate_classifier(model, loader, device, pitch_loss_weight=2.0):
    model.eval()
    running_loss = 0.0
    running_note_loss = 0.0
    running_pitch_loss = 0.0
    note_correct = 0
    pitch_correct = 0
    total = 0

    for batch in tqdm(loader, desc="classifier eval"):
        batch = move_batch(batch, device)
        outputs = model(batch["spectrogram"])
        note_loss = F.cross_entropy(outputs["note_logits"], batch["note_class"])
        pitch_loss = F.cross_entropy(outputs["pitch_logits"], batch["pitch"])
        loss = note_loss + pitch_loss_weight * pitch_loss

        batch_size = batch["note_class"].shape[0]
        running_loss += loss.item() * batch_size
        running_note_loss += note_loss.item() * batch_size
        running_pitch_loss += pitch_loss.item() * batch_size
        note_correct += (outputs["note_logits"].argmax(dim=-1) == batch["note_class"]).sum().item()
        pitch_correct += (outputs["pitch_logits"].argmax(dim=-1) == batch["pitch"]).sum().item()
        total += batch_size

    return {
        "loss": running_loss / total,
        "note_loss": running_note_loss / total,
        "pitch_loss": running_pitch_loss / total,
        "note_accuracy": note_correct / total,
        "pitch_accuracy": pitch_correct / total,
    }

In [10]:
best_note_accuracy = 0.0
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_classifier_one_epoch(model, loaders["train"], optimizer, device, PITCH_LOSS_WEIGHT)
    test_metrics = evaluate_classifier(model, loaders["test"], device, PITCH_LOSS_WEIGHT)
    best_note_accuracy = max(best_note_accuracy, test_metrics["note_accuracy"])
    print({"epoch": epoch, "train": train_metrics, "test": test_metrics})
    save_checkpoint(CHECKPOINT_PATH, model_state_dict=model.state_dict(), mel_config=mel_config, note_names=NOTE_NAMES)

best_note_accuracy

classifier train:   0%|          | 0/64 [00:00<?, ?it/s]

IndexError: too many indices for tensor of dimension 4